# Практика 05. Генерация текстов

In [45]:
import numpy as np
import pandas as pd
import tensorflow as tf

import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from keras import layers
from collections import Counter
import re
from scipy.sparse import csr_matrix

In [46]:
import keras
print("Версия Keras:", keras.__version__)

Версия Keras: 2.5.0


In [68]:
# 1. Загрузка данных
df = pd.read_csv('poems.csv', encoding='utf-8')  # или 'windows-1251' если utf-8 не работает
texts = df['text'].fillna('').to_list()  # объединяем все стихи в один текст
print (len(texts))
texts = texts[:5000] #ограничиваем количество текстов в выборке
print (len(texts))

19316
5000


In [69]:
# 2. Предобработка текста
def clean_text(text):
    # Приводим к нижнему регистру
    text = text.lower()
    # Удаляем специальные символы, но сохраняем переносы строк и основные знаки препинания
    text = re.sub(r'[^а-яё\s\n\.,!?;:—\-"]', '', text)
    # Заменяем несколько переносов строки на один
    text = re.sub(r'\n+', '\n', text)
    # Заменяем несколько пробелов на один
    text = re.sub(r' +', ' ', text)
    return text

cleaned_texts = [None] * len(texts)
for i, text in enumerate(texts):
    cleaned_texts[i] = clean_text(text)

In [70]:
cleaned_texts[1]

'на серебряные шпоры\nя в раздумии гляжу;\nза тебя, скакун мой скорый,\nза бока твои дрожу.\nнаши предки их не знали\nи, гарцуя средь степей,\nтолстой плеткой погоняли\nнедоезжаных коней.\nно с успехом просвещенья\nвместо грубой старины\nвведены изобретенья\nчужеземной стороны;\nв наше время кормят, холят,\nберегут спинную честь\nпрежде били нынче колют!..\nчто же выгодней? бог весть!..'

In [71]:
# 2. Разбиваем текст на слова с сохранением информации о строках
word_sequences = []

for text in cleaned_texts:
    # Разбиваем строку на слова, сохраняя знаки препинания как отдельные токены
    words = re.findall(r'\w+|[.,!?;:—\-"\n]', text)
    if len(words) > 1:  # Игнорируем строки с одним словом
        word_sequences.append(words)

In [72]:
# 3. Создаем словарь слов
tokenizer = Tokenizer(filters='', oov_token='<OOV>')
tokenizer.fit_on_texts([' '.join(seq) for seq in word_sequences])
word_index = tokenizer.word_index
index_word = {v: k for k, v in word_index.items()}

vocab_size = len(word_index) + 1
print(f"Размер словаря: {vocab_size} слов")
print("Примеры слов:", list(word_index.items())[:10])

Размер словаря: 126615 слов
Примеры слов: [('<OOV>', 1), ('\n', 2), (',', 3), ('.', 4), ('и', 5), ('в', 6), ('!', 7), ('—', 8), ('не', 9), ('я', 10)]


In [73]:
# 4. Подготовка последовательностей для обучения
seq_length = 17  # длина последовательности в словах
sequences = []
next_words = []

for seq in word_sequences:
    if len(seq) > seq_length:
        for i in range(0, len(seq) - seq_length):
            sequences.append(seq[i:i + seq_length])
            next_words.append(seq[i + seq_length])

print(f"Создано последовательностей: {len(sequences)}")

Создано последовательностей: 1654913


In [74]:
# 5. Оптимизированная векторизация данных (без создания полного one-hot массива)

# Использование sparse матрицы для y
X = np.zeros((len(sequences), seq_length), dtype='int32')
y_indices = np.zeros(len(sequences), dtype='int32')

for i, seq in enumerate(sequences):
    for t, word in enumerate(seq):
        X[i, t] = word_index.get(word, word_index['<OOV>'])
    y_indices[i] = word_index.get(next_words[i], word_index['<OOV>'])

# Преобразуем в sparse one-hot
y_sparse = csr_matrix((np.ones(len(y_indices)), 
                      (np.arange(len(y_indices)), y_indices)),
                      shape=(len(y_indices), vocab_size))

print("Форма X:", X.shape)
print("Форма y_sparse:", y_sparse.shape)

Форма X: (1654913, 17)
Форма y_sparse: (1654913, 126615)


In [75]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import gc

# Оптимизация использования памяти
X = X.astype('int32')
y_indices = y_indices.astype('int32')
gc.collect()

# Ограничение количества примеров если данных слишком много
MAX_SAMPLES = 70000
if len(X) > MAX_SAMPLES:
    print(f"Берём подмножество {MAX_SAMPLES} примеров")
    X = X[:MAX_SAMPLES]
    y_indices = y_indices[:MAX_SAMPLES]

Берём подмножество 70000 примеров


In [77]:
# Параметры модели
embedding_dim = 256
lstm_units = 192
batch_size = 128
epochs = 10

# Создание модели 
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim), #input_length=seq_length
    LSTM(lstm_units),
    Dense(vocab_size, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [78]:
# Callbacks
callbacks = [
    EarlyStopping(patience=2, monitor='val_loss'),
    ModelCheckpoint('best_poem_model.h5', save_best_only=True)
]

In [79]:
# Разделение на тренировочный и валидационный наборы

val_size = min(int(0.1 * len(X)), 10000)
X_train, X_val = X[:-val_size], X[-val_size:]
y_train, y_val = y_indices[:-val_size], y_indices[-val_size:]

In [80]:
import tensorflow as tf

print(tf.config.list_physical_devices())

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [81]:
# Настройка GPU
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    for device in physical_devices:
        tf.config.experimental.set_memory_growth(device, True)
else:
    print("No GPUs found.")

No GPUs found.


In [82]:
# Обучение модели
history = model.fit(
    X_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    workers=-1,
    use_multiprocessing=True
)

# Сохранение модели
model.save('poem_generator_final.h5')
print("Обучение завершено. Модель сохранена как 'poem_generator_final.h5'")

Epoch 1/10
493/493 [==============================] - 402s 813ms/step - loss: 7.2242 - accuracy: 0.1607 - val_loss: 6.9288 - val_accuracy: 0.2101
Epoch 2/10
493/493 [==============================] - 407s 825ms/step - loss: 6.0198 - accuracy: 0.2128 - val_loss: 6.8199 - val_accuracy: 0.2240
Epoch 3/10
493/493 [==============================] - 409s 829ms/step - loss: 5.6999 - accuracy: 0.2305 - val_loss: 6.8944 - val_accuracy: 0.2269
Epoch 4/10
493/493 [==============================] - 408s 828ms/step - loss: 5.4848 - accuracy: 0.2379 - val_loss: 6.9497 - val_accuracy: 0.2271
Обучение завершено. Модель сохранена как 'poem_generator_final.h5'


In [83]:
# После обучения модели, перед генерацией:
np.save('word_index.npy', word_index)
np.save('index_word.npy', index_word)

In [84]:
import random
from tensorflow.keras.models import load_model
from collections import defaultdict

class PoemGenerator:
    def __init__(self, model_path, word_index_path, index_word_path):
        self.model = load_model(model_path)
        self.word_index = np.load(word_index_path, allow_pickle=True).item()
        self.index_word = np.load(index_word_path, allow_pickle=True).item()
        self.vocab_size = len(self.word_index)
        
        # Словарь рифм (можно расширить)
        self.rhymes = {
            'ом': ['дом', 'сом', 'ком', 'льдом'],
            'ай': ['край', 'рай', 'май', 'дай'],
            'ец': ['отец', 'певец', 'горец', 'конец'],
            'ая': ['красная','красивая','милая','ленивая'],
            'а': ['душа','спеша','суша','Маша'],
            'ья': ['друзья','судья','свинья','семья'],
            'оль': ['боль','король','соль','роль'],
            'ывший': ['бывший','прибывший','отплывший','укрывший'],
            'ир': ['мир','сир','пир','сыр'],
            'он': ['трон','сон','стон','слон']
        }
        
    def _prepare_input(self, tokens, seq_length):
        """Подготовка входной последовательности"""
        if len(tokens) < seq_length:
            tokens = ['<PAD>']*(seq_length-len(tokens)) + tokens
        else:
            tokens = tokens[-seq_length:]
        return [self.word_index.get(word, self.word_index['<OOV>']) for word in tokens]

    def _find_rhyme(self, word):
        """Поиск рифмы к слову"""
        for ending, rhymes in self.rhymes.items():
            if word.endswith(ending):
                return random.choice(rhymes)
        return None

    def _select_next_word(self, preds, temperature, current_line, rhyme_target=None):
        """Выбор следующего слова с учетом рифмы"""
        preds = np.log(np.maximum(preds, 1e-10)) / temperature
        exp_preds = np.exp(preds)
        preds = exp_preds / np.sum(exp_preds)
        
        # Усиление вероятности рифмующихся слов
        if rhyme_target and random.random() > 0.3:
            for idx, word in self.index_word.items():
                if word and self._find_rhyme(word) == rhyme_target:
                    preds[idx] *= 10
        
        return random.choices(range(len(preds)), weights=preds)[0]

    def generate_poem(self, seed, theme, num_lines=8, temperature=0.7, seq_length=20):
        """Генерация стихотворения со структурой"""
        lines = []
        current_line = []
        rhyme_scheme = ['A', 'B', 'A', 'B', 'C', 'D', 'C', 'D']  # Схема рифмовки
        rhyme_words = {}  # Слова для рифм
        
        tokens = re.findall(r'\w+|[.,!?;—"\n]', seed.lower())
        
        for line_num in range(num_lines):
            for _ in range(15):  # Максимальная длина строки
                # Подготовка входных данных
                input_seq = self._prepare_input(tokens, seq_length)
                input_seq = np.array(input_seq).reshape(1, -1)
                
                # Предсказание
                preds = self.model.predict(input_seq, verbose=0)[0]
                
                # Выбор следующего слова
                next_idx = self._select_next_word(
                    preds, 
                    temperature,
                    current_line,
                    rhyme_words.get(rhyme_scheme[line_num])
                )
                next_word = self.index_word.get(next_idx, '')
                
                # Проверка на конец строки
                if next_word in '\n.!?—' or len(current_line) >= 8:
                    if len(current_line) >= 3:  # Минимальная длина строки
                        # Сохраняем последнее слово для рифмы
                        if rhyme_scheme[line_num] not in rhyme_words:
                            rhyme_word = current_line[-1].lower()
                            rhyme_words[rhyme_scheme[line_num]] = self._find_rhyme(rhyme_word) or rhyme_word
                        break
                elif next_word not in '.,!?;—"\n':
                    current_line.append(next_word)
                
                tokens.append(next_word)
                if len(tokens) > seq_length * 2:
                    tokens = tokens[-seq_length:]
            
            # Форматирование строки
            line = ' '.join(current_line).capitalize()
            if line_num < num_lines:
                lines.append(line)
            current_line = []
        
        # Пост-обработка
        poem = '\n'.join(lines)
        poem = re.sub(r' ([.,!?;—])', r'\1', poem)  # Убираем пробелы перед знаками препинания
        return poem

# Пример использования
generator = PoemGenerator(
    model_path='poem_generator_final.h5',
    word_index_path='word_index.npy',
    index_word_path='index_word.npy'
)

poem = generator.generate_poem(
    seed="Осень золотая",
    theme="осень",
    num_lines=8,
    temperature=0.8
)

In [85]:
#больше примеров, сама остановилась на 4 эпохах
print(poem)

И может воскликнул
: в не горами в
Только он не горы я кров
И хаджи чтобы стыд в жизнь мою
На любезный лесов
Что бедный взошло
Я во вспоминая
Где мог зовут он давно


In [67]:
print(poem) # 4 эпохи, сама остановилась

Что играя не как вы и не неприступной
На кудри своей
Как нам я не вьюга
Над потом печали
Ее стыдясь и поток
И от добро вдруг пир
Лишь люди путников в горит
Что с нем ее моего


In [65]:
print(poem) # 4 эпохи, сама остановилась

И не в ним как как ты что
Как и что в нем как не вновь
И в землю как не сон
И в ним и и не нем
И в земле как не море
И в ним он и не ней
И в земле как в ним
И не первый и не нем


In [63]:
print(poem) # 4 эпохи, сама остановилась

И на землю и в тени как я
Что не катится он в ним
И что под ним и земле
И как в тех пор
И что так - в ним и простой
В путь нас в них
Издалека в ним мне как ты в ним
Не думал но я


In [44]:
print(poem) # 3 эпохи

И грустию осени руку жестокой
: когда верит в кровь облак
И глуши наряд казака
Не покормлю и бы
Он мы они кровь
В скучен груди ничего
Не тобою как любил их целый
И сам он взор мы тут


In [22]:
print(poem)

Руку и все ж со голове и поблизости
И дорис - желания что на небу
Дафна о насмешкой приятно как еще счастья
Смерть я с собой
Зачем мы оставляя и рожден
В которой живой но хоть он не мог
Же нам музыки и не медли ручить
И иксиона что грех не надо


In [24]:
poem2 = generator.generate_poem(
    seed="Первое мая",
    theme="весна",
    num_lines=6,
    temperature=0.7
)
print (poem2)

Корифей и дочь моя
Склонившись к нему как и мы
Я в молчаньи пустом у цели нет
: а это ты что ж мне один
В досада сей отец и отдать
Тебе что ж не скажут : горе


In [93]:
poem2 = generator.generate_poem(
    seed="победа",
    theme="весна",
    num_lines=4,
    temperature=0.6
)
print (poem2)

И жизнь за как вздрогнул
И не груди не ним мне не родной
И не как как то в ним
И словом лезгинец


## Практическое задание.
Необходимо внести исправления либо в обработку данных, либо в структуру и обучение НС, чтобы обучение не деградировало, а постоянно улучшалось.